In [3]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from importlib import reload
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression
import numpy as np
import statsmodels.api as sm  # module to build a LOWESS model
import piecewise_regression

In [4]:
sys.path.append("../../src")

In [5]:
import main

# Meteobikes

Meteobike data from 27th June 2023

In [6]:
mbike = pd.read_csv('../../data/raw_data/ALL-SYSTEMS-2023-06-27.csv')

In [7]:
mbike['WRTM_temp'] = mbike['Absolute_temperature_degC'] - mbike['Temperature_diff_K']

In [ ]:
mbike['WRTM_temp'].mean()

In [ ]:
mbike['Absolute_temperature_degC'].mean()

# If running for the first time

In [8]:
bldgs = gpd.read_parquet('../../data/processed_data/processed_buildings_3D.parquet')
streets = gpd.read_parquet('../../data/processed_data/processed_streets.parquet')
nodes = gpd.read_parquet('../../data/processed_data/processed_nodes.parquet')

In [1]:
# up to radius 700m, every mbike measurement was takes
# radius 800-900m, every 5th measurement was taken
# radius 1000m, every 10th measurement was taken
# radius 1000-1200m, every 15th measurement was taken
# radius 1500, every 30th measurement was taken
# radius 2000, every 100th measurement was taken
# radius 3000, every 200th measurement was taken

radii = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200,210,220,230,240,250,260,270,280,290,300,320,340,360,380,400,450,500,600,750,1000,1250,1500,1750,2000]


In [ ]:
radii = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200,210,220,230,240,250,260,270,280,290]
for i in radii:
    stn_buffers = main.buffer_stations(mbike, radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs, stn_buffers)
    b,s,n = main.select_objects(bldgs, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers, three_d=True)

    # replace all NaNs in stn_buffers['BuAre_sum'] with 0
    stn_buffers['BuAre_sum'] = stn_buffers['BuAre_sum'].fillna(0)
    stn_buffers['BuVol_3D_sum'] = stn_buffers['BuVol_3D_sum'].fillna(0)
    stn_buffers['BuEWA_3D_sum'] = stn_buffers['BuEWA_3D_sum'].fillna(0)

    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_' + str(i) + '.parquet')

up to radius 400 every mbike
400-600 every 5 measurements
750 every 10m
1000m every 20m
1250m every 50
1500 m every 100
1750, 2000 every 200

In [ ]:
radii = [1500]

for i in radii:
    stn_buffers = main.buffer_stations(mbike[::100], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs, stn_buffers)
    b,s,n = main.select_objects(bldgs, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers, three_d=True)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_' + str(i) + '.parquet')

In [ ]:
radii = [1750, 2000]

for i in radii:
    stn_buffers = main.buffer_stations(mbike[::200], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs, stn_buffers)
    b,s,n = main.select_objects(bldgs, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers, three_d=True)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_' + str(i) + '.parquet')

In [13]:
bldgs_merged = gpd.read_parquet('../../data/processed_data/processed_buildings_merged_blocks.parquet')

In [ ]:
# for merged blocks
radii = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200,210,220,230,240,250,260,270,280,290,300,320,340,360,380,400]
for i in radii:
    stn_buffers = main.buffer_stations(mbike, radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [450,500,600]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::5], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [750]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::10], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [1000]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::20], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [1250]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::50], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [1500]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::100], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

In [ ]:
# for merged blocks
radii = [1750,2000]
for i in radii:
    stn_buffers = main.buffer_stations(mbike[::200], radius=i, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/270623_processed_station_params_merged_' + str(i) + '.parquet')

# If running again

In [10]:
radius = 300
stn_buffers = gpd.read_parquet(f'../../data/processed_data/mbike/270623_processed_station_params_{radius}.parquet')
stn_buffers_merged = gpd.read_parquet(f'../../data/processed_data/mbike/270623_processed_station_params_merged_{radius}.parquet')

In [11]:
stn_buffers['WRTM_temp'] = stn_buffers['Absolute_temperature_degC'] - stn_buffers['Temperature_diff_K']
stn_buffers['temp_no_cooling'] = stn_buffers['Temperature_diff_K'] + stn_buffers['WRTM_temp'].mean()

In [12]:
stn_buffers['BuAre_sum'] = stn_buffers['BuAre_sum'] / (radius**2*np.pi)
stn_buffers['BuEWA_3D_sum'] = stn_buffers['BuEWA_3D_sum'] / (radius**2*np.pi)
stn_buffers["BuVol_3D_sum"] = stn_buffers["BuVol_3D_sum"] / (radius**2*np.pi)

stn_buffers['HW_del_merged'] = stn_buffers['BuHt_wmean'] / stn_buffers_merged['BuW_delaunay_mean']
stn_buffers['HW_knn1_merged'] = stn_buffers['BuHt_wmean'] / stn_buffers_merged['BuW_knn1_mean']
stn_buffers['HW_knn5_merged'] = stn_buffers['BuHt_wmean'] / stn_buffers_merged['BuW_knn5_mean']

stn_buffers['BuW_delaunay_mean_merged'] = stn_buffers_merged['BuW_delaunay_mean']
stn_buffers['BuW_knn5_mean_merged'] = stn_buffers_merged['BuW_knn5_mean']

stn_buffers['BuERI_mode_merged'] = stn_buffers_merged['BuERI_mode']
stn_buffers['BuERI_wmean_merged'] = stn_buffers_merged['BuERI_wmean']

stn_buffers['HW_geo'] = stn_buffers['BuEWA_3D_sum'] / (1 - stn_buffers['BuAre_sum'])

In [13]:
# add AWS temp to plot

temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')

In [ ]:
stn_buffers['Temperature_diff_K'].std()

In [15]:
# add AWS morphometric parameters to plot

radius = 300
vars = gpd.read_parquet(f'../../data/processed_data/processed_station_params_{radius}.parquet')
vars_merged = gpd.read_parquet(f'../../data/processed_data/processed_station_params_merged_blocks_{radius}.parquet')
vars.index = vars['station_id']
vars_merged.index = vars_merged['station_id']
to_remove = ['station_id','station_no','station_name','station_long_name','station_type','station_lat','station_lon','station_elevation','mounting_structure','sky_view_factor','dominant_land_use','local_climate_zone','urban_atlas_class','urban_atlas_code','geometry','SVF_3D','station_elevation_diff']
vars = vars.drop(to_remove, axis=1)

vars['BuAre_sum'] = vars['BuAre_sum'] / (radius**2*np.pi)
vars['BuEWA_3D_sum'] = vars['BuEWA_3D_sum'] / (radius**2*np.pi)
vars["BuVol_3D_sum"] = vars["BuVol_3D_sum"] / (radius**2*np.pi)

vars['HW_del_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_delaunay_mean']
vars['HW_knn1_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn1_mean']
vars['HW_knn5_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn5_mean']

vars['BuW_delaunay_mean_merged'] = vars_merged['BuW_delaunay_mean']
vars['BuW_knn5_mean_merged'] = vars_merged['BuW_knn5_mean']

vars['BuERI_mode_merged'] = vars_merged['BuERI_mode']
vars['BuERI_wmean_merged'] = vars_merged['BuERI_wmean']

temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')
temp.index = temp.index.str[2:]

vars = vars.merge(temp, left_on='station_id', right_on='station_id',how='inner')

In [16]:
# for all params
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
correlation_results = {}
for param in params:
    tv = vars[[param,'2023-06-27 20:00:00+00:00']].dropna()
    spearman_corr, spearman_p = spearmanr(tv[param], tv['2023-06-27 20:00:00+00:00'])
    correlation_results[param] = {'spearman_corr': spearman_corr, 'spearman_p': spearman_p}

In [ ]:
correlation_results

In [ ]:
var_name_mapping = {
    'BuAre_sum': 'Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',
    'BuVol_3D_sum': 'Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',
    'BuEWA_3D_sum': 'Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)', 
    'BuAdj': 'Adjacency ($\it{Adj}$)',
    'BuHt_wmean': 'Height ($\it{H}$, m)',
    'SVF_3D_mean': 'Sky View Factor ($\it{SVF}$)',
    'BuERI_mode_merged': 'Shape Complexity ($\it{SC}$)',
    'StrClo400_median': 'Street Network Connectivity ($\it{C}$)',
    'StrHW_wmean': 'Height:Width v3 ($\it{H:W_3}$)',
    'HW_geo':'Height:Width v2 ($\it{H:W_2}$)',
    'HW_del_merged':'Height:Width v1 ($\it{H:W_1}$)',
    'BuW_delaunay_mean_merged':'Width ($\it{W}$, m)',
    'BuW_delaunay_mean':'Width (cadastre) ($\it{W}$, m)',
    'HW_knn1_merged':'HW_knn1_merged',
    'HW_knn5_merged':'HW_knn5_merged',
    'NdDen':'$\it{ND}$'
    }

In [19]:
# lowess
lowess = sm.nonparametric.lowess

def calculate_slopes(temp_lowess, turn_a, turn_b, x_0=None, x_max=None):
    
    # nearest LOESS point to breakpoint
    idx_a = np.argmin(np.abs(temp_lowess[:,0] - turn_a))
    idx_b = np.argmin(np.abs(temp_lowess[:,0] - turn_b))

    x_turn_a, y_turn_a = temp_lowess[idx_a]
    x_turn_b, y_turn_b = temp_lowess[idx_b]

    if x_0 is None:
        x_0, y_0 = temp_lowess[0]
    else:
        y_0 = np.interp(x_0, temp_lowess[:,0], temp_lowess[:,1])
    if x_max is None:
        x_max, y_max = temp_lowess[-1]
    else:
        y_max = np.interp(x_max, temp_lowess[:,0], temp_lowess[:,1])

    slope1 = (y_turn_a - y_0) / (x_turn_a - x_0)
    slope2 = (y_max - y_turn_b) / (x_max - x_turn_b)

    return slope1, slope2

### Check sensitivity of meteobike lag (shifting measurement by approx. 30 s) to account for the air temperature sensor lag

In [96]:
# Make sure the data are ordered correctly within each system
df = stn_buffers.sort_values(['System_ID', 'Seconds_in_day']).copy()

# Shift BuAre_sum by 5 measurements within each System_ID
df['BuAre_sum_lag5'] = (
    df.groupby('System_ID')['BuAre_sum']
      .shift(3)
)

# Drop rows where there is no corresponding measurement 5 positions later
df = df.dropna(subset=['BuAre_sum_lag5']).copy()

In [ ]:
stn_buffers[['System_ID', 'Time_UTC', 'Seconds_in_day', 'BuAre_sum', 'Temperature_diff_K']].head(10)

In [ ]:
plt.hexbin(df['BuAre_sum_lag5'], df['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)

In [ ]:
spearmanr(df['BuAre_sum_lag5'], df['temp_no_cooling']+0.2)

In [ ]:
data = df[['BuAre_sum_lag5','temp_no_cooling']].dropna()
x = data['BuAre_sum_lag5'].values
y = data['temp_no_cooling'].values + 0.2
pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
print(pw_fit.summary())
pw_fit.plot_fit(color="red", linewidth=2)

## Make Plots

In [ ]:
# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum', 'BuIBD', 'BuAdj', 'BuSWR_3D_median', 'BuHt_wmean','StrHW_median','SVF_3D_mean', 'BuERI_mode','StrClo400_median']
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(17, 22))
gs = gridspec.GridSpec(4, 3, figure=fig, wspace=0.05, hspace=0.3)  # 4 rows, 3 columns

#fig.subplots_adjust(wspace=0.1)
for i in range(12):  # First 12 subplots
    ax = fig.add_subplot(gs[i])

    if params[i] == 'BuAre_sum':
        im= ax.hexbin(stn_buffers['BuAre_sum'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuAre_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.76""\n""$ρ_{mob,300,T}$: 0.71""\n""$MI_{mob,300,T}$: 0.75", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuAre_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.17, turn_b=0.25))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuVol_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['BuVol_3D_sum'], stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuVol_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.77""\n""$ρ_{mob,300,T}$: 0.72""\n""$MI_{mob,300,T}$: 0.84", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuVol_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=1.6, turn_b=3.0))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuEWA_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['BuEWA_3D_sum'], stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuEWA_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.66""\n""$ρ_{mob,300,T}$: 0.68""\n""$MI_{mob,300,T}$: 0.74", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuEWA_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.3, turn_b=0.75))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuHt_wmean':
        ax.hexbin(stn_buffers[stn_buffers['BuHt_wmean']!=0]['BuHt_wmean'], stn_buffers[stn_buffers['BuHt_wmean']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars[vars['BuHt_wmean']>1]['BuHt_wmean'], vars[vars['BuHt_wmean']>1]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height ($\it{H}$, m)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.72""\n""$ρ_{mob,300,T}$: 0.58""\n""$MI_{mob,300,T}$: 0.69", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuHt_wmean'], is_sorted=False, frac =0.7)
        print(params[i], calculate_slopes(temp_lowess, turn_a=6, turn_b=7, x_max=15))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuW_delaunay_mean_merged':
        ax.hexbin((stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['BuW_delaunay_mean_merged']), stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter((vars['BuW_delaunay_mean_merged']), vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Width ($\it{W}$, m)',fontsize=16)
        #ax.set_xscale('log')
        ax.set_xlim(0,250)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.15""\n""$ρ_{mob,300,T}$: -0.28""\n""$MI_{mob,300,T}$: 0.86", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))      
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuW_delaunay_mean_merged'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=150, turn_b=170))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'SVF_3D_mean':
        ax.hexbin(stn_buffers['SVF_3D_mean'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['SVF_3D_mean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Sky View Factor ($\it{SVF}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.64""\n""$ρ_{mob,300,T}$: -0.71""\n""$MI_{mob,300,T}$: 0.78", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['SVF_3D_mean'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.8, turn_b=0.9))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)

    elif params[i] == 'HW_del_merged':
        ax.hexbin(stn_buffers[stn_buffers['HW_del_merged']!=0]['HW_del_merged'], stn_buffers[stn_buffers['HW_del_merged']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_del_merged'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v1 ($\it{H:W_1}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.45""\n""$ρ_{mob,300,T}$: 0.66""\n""$MI_{mob,300,T}$: 0.79", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))   
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_del_merged'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.5, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'HW_geo':
        ax.hexbin(stn_buffers[stn_buffers['HW_geo']!=0]['HW_geo'], stn_buffers[stn_buffers['HW_geo']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_geo'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v2 ($\it{H:W_2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.69""\n""$ρ_{mob,300,T}$: 0.69""\n""$MI_{mob,300,T}$: 0.76", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_geo'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.4, turn_b=0.8))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
    
    elif params[i] == 'StrHW_wmean':
        ax.hexbin(stn_buffers[stn_buffers['StrHW_wmean']<10]['StrHW_wmean'],stn_buffers[stn_buffers['StrHW_wmean']<10]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrHW_wmean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v3 ($\it{H:W_3}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.61""\n""$ρ_{mob,300,T}$: 0.51""\n""$MI_{mob,300,T}$: 0.78", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrHW_wmean'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.6, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        #ax.set_xlim(0,1.5)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)

    elif params[i] == 'BuERI_mode_merged':
        # swap order so that low ERI is high SC 
        ax.hexbin(1-stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['BuERI_mode_merged'], stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['BuERI_mode_merged'], vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Shape Complexity ($\it{SC}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.68""\n""$ρ_{mob,300,T}$: -0.46""\n""$MI_{mob,300,T}$: 1.41", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuERI_mode_merged'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.45, turn_b=0.55))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = (1-data[params[i]]).values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)

    elif params[i] == 'BuAdj':
        ax.hexbin(1-stn_buffers['BuAdj'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars['BuAdj'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Adjacency ($\it{Adj}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.63""\n""$ρ_{mob,300,T}$: 0.61""\n""$MI_{mob,300,T}$: 0.76", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuAdj'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.42, turn_b=0.52))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = (1-data[params[i]]).values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'StrClo400_median':
        ax.hexbin(stn_buffers['StrClo400_median'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrClo400_median'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Street Network Connectivity ($\it{C}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.29""\n""$ρ_{mob,300,T}$: 0.40""\n""$MI_{mob,300,T}$: 0.80", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrClo400_median'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.000008, turn_b=0.000013))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)


    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel(r'$\it{T_a}$ ($\degree$C)',fontsize=16)
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks
    
# Add a colorbar below the entire plot
cbar_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', drawedges=False)

# Customize the colorbar
cbar.ax.tick_params(axis='x', labelsize=14)  # Adjust tick font size
cbar.ax.set_xlabel('Count', fontsize=16)  # Set the label

plt.tight_layout(w_pad=0.01, h_pad=0.3)

plt.savefig('../../figures/fig6/fig6_segmented.png', dpi=650, bbox_inches='tight')
plt.savefig('../../figures/fig6/fig6_segmented.pdf')
plt.savefig('../../figures/fig6/fig6_segmented.svg')
plt.show()

In [ ]:
# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum', 'BuIBD', 'BuAdj', 'BuSWR_3D_median', 'BuHt_wmean','StrHW_median','SVF_3D_mean', 'BuERI_mode','StrClo400_median']
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(17, 22))
gs = gridspec.GridSpec(4, 3, figure=fig, wspace=0.05, hspace=0.3)  # 4 rows, 3 columns

#fig.subplots_adjust(wspace=0.1)
for i in range(12):  # First 12 subplots
    ax = fig.add_subplot(gs[i])

    if params[i] == 'BuAre_sum':
        im= ax.hexbin(stn_buffers['BuAre_sum'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuAre_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.76""\n""$ρ_{mob,300,T}$: 0.71""\n""$MI_{mob,300,T}$: 0.75", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuAre_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.17, turn_b=0.25))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuVol_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['BuVol_3D_sum'], stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuVol_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.77""\n""$ρ_{mob,300,T}$: 0.72""\n""$MI_{mob,300,T}$: 0.84", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuVol_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=1.6, turn_b=3.0))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuEWA_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['BuEWA_3D_sum'], stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuEWA_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.66""\n""$ρ_{mob,300,T}$: 0.68""\n""$MI_{mob,300,T}$: 0.74", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuEWA_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.3, turn_b=0.75))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'BuHt_wmean':
        ax.hexbin(stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['BuHt_wmean'], stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars[vars['BuHt_wmean']>1]['BuHt_wmean'], vars[vars['BuHt_wmean']>1]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height ($\it{H}$, m)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.72""\n""$ρ_{mob,300,T}$: 0.58", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuHt_wmean'], is_sorted=False, frac =0.7)
        print(params[i], calculate_slopes(temp_lowess, turn_a=6, turn_b=7, x_max=15))
        ax.plot(temp_lowess[:-20, 0], temp_lowess[:-20, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        X = sm.add_constant(data[params[i]].values)  # Add constant for regression
        model = sm.OLS(data['temp_no_cooling'].values + 0.2, X).fit()
        # Get the predicted values (fitted values)
        y_pred = model.fittedvalues
        ax.plot(data[params[i]].values[:-4900], y_pred[:-4900], color='grey', linewidth=1.5)
        
    elif params[i] == 'BuW_delaunay_mean_merged':
        ax.hexbin((stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['BuW_delaunay_mean_merged']), stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter((vars['BuW_delaunay_mean_merged']), vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Width ($\it{W}$, m)',fontsize=16)
        #ax.set_xscale('log')
        ax.set_xlim(0,250)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.15""\n""$ρ_{mob,300,T}$: -0.28""\n""$MI_{mob,300,T}$: 0.86", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))      
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuW_delaunay_mean_merged'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=150, turn_b=170))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[data[params[i]]<250][params[i]].values
        y = data[data[params[i]]<250]['temp_no_cooling'].values + 0.2
        X = sm.add_constant(x)  # Add constant for regression
        model = sm.OLS(y, X).fit()
        # Get the predicted values (fitted values)
        y_pred = model.fittedvalues
        ax.plot(x, y_pred, color='grey', linewidth=1.5)
        
    elif params[i] == 'SVF_3D_mean':
        ax.hexbin(stn_buffers['SVF_3D_mean'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['SVF_3D_mean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Sky View Factor ($\it{SVF}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.64""\n""$ρ_{mob,300,T}$: -0.71""\n""$MI_{mob,300,T}$: 0.78", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['SVF_3D_mean'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.8, turn_b=0.9))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)

    elif params[i] == 'HW_del_merged':
        ax.hexbin(stn_buffers[stn_buffers['HW_del_merged']!=0]['HW_del_merged'], stn_buffers[stn_buffers['HW_del_merged']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_del_merged'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v1 ($\it{H:W_1}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.45""\n""$ρ_{mob,300,T}$: 0.66""\n""$MI_{mob,300,T}$: 0.79", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))   
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_del_merged'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.5, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
        
    elif params[i] == 'HW_geo':
        ax.hexbin(stn_buffers[stn_buffers['HW_geo']!=0]['HW_geo'], stn_buffers[stn_buffers['HW_geo']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_geo'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v2 ($\it{H:W_2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.69""\n""$ρ_{mob,300,T}$: 0.69""\n""$MI_{mob,300,T}$: 0.76", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_geo'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.4, turn_b=0.8))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)
    
    elif params[i] == 'StrHW_wmean':
        ax.hexbin(stn_buffers[stn_buffers['StrHW_wmean']<10]['StrHW_wmean'],stn_buffers[stn_buffers['StrHW_wmean']<10]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrHW_wmean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v3 ($\it{H:W_3}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.61""\n""$ρ_{mob,300,T}$: 0.51""\n""$MI_{mob,300,T}$: 0.78", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrHW_wmean'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.6, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        #ax.set_xlim(0,1.5)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)

    elif params[i] == 'BuERI_mode_merged':
        # swap order so that low ERI is high SC 
        ax.hexbin(1-stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['BuERI_mode_merged'], stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['BuERI_mode_merged'], vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Shape Complexity ($\it{SC}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.68""\n""$ρ_{mob,300,T}$: -0.46""\n""$MI_{mob,300,T}$: 1.41", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuERI_mode_merged'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.45, turn_b=0.55))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = (1-data[params[i]]).values
        y = data['temp_no_cooling'].values + 0.2
        X = sm.add_constant(x)  # Add constant for regression
        model = sm.OLS(y, X).fit()
        # Get the predicted values (fitted values)
        y_pred = model.fittedvalues
        ax.plot(x, y_pred, color='grey', linewidth=1.5)

    elif params[i] == 'BuAdj':
        ax.hexbin(1-stn_buffers['BuAdj'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars['BuAdj'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Adjacency ($\it{Adj}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.63""\n""$ρ_{mob,300,T}$: 0.61""\n""$MI_{mob,300,T}$: 0.76", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuAdj'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.42, turn_b=0.52))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = (1-data[params[i]]).values
        y = data['temp_no_cooling'].values + 0.2
        X = sm.add_constant(x)  # Add constant for regression
        model = sm.OLS(y, X).fit()
        # Get the predicted values (fitted values)
        y_pred = model.fittedvalues
        ax.plot(x, y_pred, color='grey', linewidth=1.5)
        
    elif params[i] == 'StrClo400_median':
        ax.hexbin(stn_buffers['StrClo400_median'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrClo400_median'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Street Network Connectivity ($\it{C}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.29""\n""$ρ_{mob,300,T}$: 0.40""\n""$MI_{mob,300,T}$: 0.80", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrClo400_median'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.000008, turn_b=0.000013))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

        data = stn_buffers[[params[i],'temp_no_cooling']].dropna()
        x = data[params[i]].values
        y = data['temp_no_cooling'].values + 0.2
        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        print(pw_fit.summary())
        pw_fit.plot_fit(color="limegreen", linewidth=2)


    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel(r'$\it{T_a}$ ($\degree$C)',fontsize=16)
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks
    
# Add a colorbar below the entire plot
cbar_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', drawedges=False)

# Customize the colorbar
cbar.ax.tick_params(axis='x', labelsize=14)  # Adjust tick font size
cbar.ax.set_xlabel('Count', fontsize=16)  # Set the label

plt.tight_layout(w_pad=0.01, h_pad=0.3)

plt.savefig('../../figures/fig6/fig6_segmented_ols.png', dpi=650, bbox_inches='tight')
plt.savefig('../../figures/fig6/fig6_segmented_ols.pdf')
plt.savefig('../../figures/fig6/fig6_segmented_ols.svg')
plt.show()

In [ ]:
data[data[params[i]]<250][params[i]].values

In [ ]:
ax.hexbin(stn_buffers[stn_buffers['HW_geo']!=0]['HW_geo'], stn_buffers[stn_buffers['HW_geo']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_geo'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v2 ($\it{H:W_2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.69""\n""$ρ_{mob,300,T}$: 0.72", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_geo'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.4, turn_b=0.8))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

In [ ]:
len(y_pred)

In [ ]:
X = sm.add_constant(data[var])  # Add constant for regression
        model = sm.OLS(data['temperature'], X).fit()
        bse = model.bse[var]
        r_squared = model.rsquared

        # Get the predicted values (fitted values)
        y_pred = model.fittedvalues

In [ ]:
# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(17, 22))
gs = gridspec.GridSpec(4, 3, figure=fig, wspace=0.05, hspace=0.3)  # 4 rows, 3 columns

#fig.subplots_adjust(wspace=0.1)
for i in range(12):  # First 12 subplots
    ax = fig.add_subplot(gs[i])

    if params[i] == 'BuAre_sum':
        im= ax.hexbin(stn_buffers['BuAre_sum'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuAre_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.76""\n""$ρ_{mob,300,T}$: 0.74", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuAre_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.17, turn_b=0.25))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuVol_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['BuVol_3D_sum'], stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuVol_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.77""\n""$ρ_{mob,300,T}$: 0.75", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuVol_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=1.6, turn_b=3.0))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuEWA_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['BuEWA_3D_sum'], stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuEWA_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.66""\n""$ρ_{mob,300,T}$: 0.71", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuEWA_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.3, turn_b=0.75))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuHt_wmean':
        ax.hexbin(stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['BuHt_wmean'], stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars[vars['BuHt_wmean']>1]['BuHt_wmean'], vars[vars['BuHt_wmean']>1]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height ($\it{H}$, m)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.72""\n""$ρ_{mob,300,T}$: 0.58", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuHt_wmean'], is_sorted=False, frac =0.7)
        print(params[i], calculate_slopes(temp_lowess, turn_a=6, turn_b=7, x_max=15))
        ax.plot(temp_lowess[:-20, 0], temp_lowess[:-20, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuW_delaunay_mean_merged':
        ax.hexbin((stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['BuW_delaunay_mean_merged']), stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter((vars['BuW_delaunay_mean_merged']), vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Width ($\it{W}$, m)',fontsize=16)
        #ax.set_xscale('log')
        ax.set_xlim(0,250)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.15""\n""$ρ_{mob,300,T}$: -0.28", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))      
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuW_delaunay_mean_merged'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=150, turn_b=170))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'SVF_3D_mean':
        ax.hexbin(stn_buffers['SVF_3D_mean'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['SVF_3D_mean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Sky View Factor ($\it{SVF}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.64""\n""$ρ_{mob,300,T}$: -0.71", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['SVF_3D_mean'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.8, turn_b=0.9))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    elif params[i] == 'HW_del_merged':
        ax.hexbin(stn_buffers[stn_buffers['HW_del_merged']!=0]['HW_del_merged'], stn_buffers[stn_buffers['HW_del_merged']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_del_merged'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v1 ($\it{H:W_1}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.45""\n""$ρ_{mob,300,T}$: 0.56", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))   
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_del_merged'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.5, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'HW_geo':
        ax.hexbin(stn_buffers[stn_buffers['HW_geo']!=0]['HW_geo'], stn_buffers[stn_buffers['HW_geo']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_geo'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v2 ($\it{H:W_2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.69""\n""$ρ_{mob,300,T}$: 0.72", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_geo'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.4, turn_b=0.8))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
    
    elif params[i] == 'StrHW_wmean':
        ax.hexbin(stn_buffers[stn_buffers['StrHW_wmean']<10]['StrHW_wmean'],stn_buffers[stn_buffers['StrHW_wmean']<10]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrHW_wmean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v3 ($\it{H:W_3}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.61""\n""$ρ_{mob,300,T}$: 0.51", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrHW_wmean'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.6, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        #ax.set_xlim(0,1.5)

    elif params[i] == 'BuERI_mode_merged':
        # swap order so that low ERI is high SC 
        ax.hexbin(1-stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['BuERI_mode_merged'], stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['BuERI_mode_merged'], vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Shape Complexity ($\it{SC}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.68""\n""$ρ_{mob,300,T}$: -0.66", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuERI_mode_merged'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.45, turn_b=0.55))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    elif params[i] == 'BuAdj':
        ax.hexbin(1-stn_buffers['BuAdj'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars['BuAdj'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Adjacency ($\it{Adj}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.63""\n""$ρ_{mob,300,T}$: 0.61", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuAdj'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.42, turn_b=0.52))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'StrClo400_median':
        ax.hexbin(stn_buffers['StrClo400_median'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrClo400_median'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Street Network Connectivity ($\it{C}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.29""\n""$ρ_{mob,300,T}$: 0.40", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrClo400_median'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.000008, turn_b=0.000013))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel(r'$\it{T_a}$ ($\degree$C)',fontsize=16)
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks
    
# Add a colorbar below the entire plot
cbar_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', drawedges=False)

# Customize the colorbar
cbar.ax.tick_params(axis='x', labelsize=14)  # Adjust tick font size
cbar.ax.set_xlabel('Count', fontsize=16)  # Set the label

plt.tight_layout(w_pad=0.01, h_pad=0.3)


plt.show()

In [ ]:
# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(17, 22))
gs = gridspec.GridSpec(4, 3, figure=fig, wspace=0.05, hspace=0.3)  # 4 rows, 3 columns

#fig.subplots_adjust(wspace=0.1)
for i in range(12):  # First 12 subplots
    ax = fig.add_subplot(gs[i])

    if params[i] == 'BuAre_sum':
        im= ax.hexbin(stn_buffers['BuAre_sum'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuAre_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.76""\n""$ρ_{mob,300,T}$: 0.74", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuAre_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.17, turn_b=0.25))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuVol_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['BuVol_3D_sum'], stn_buffers[stn_buffers['BuVol_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuVol_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.77""\n""$ρ_{mob,300,T}$: 0.75", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuVol_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=1.6, turn_b=3.0))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuEWA_3D_sum':
        ax.hexbin(stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['BuEWA_3D_sum'], stn_buffers[stn_buffers['BuEWA_3D_sum']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['BuEWA_3D_sum'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.66""\n""$ρ_{mob,300,T}$: 0.71", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuEWA_3D_sum'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.3, turn_b=0.75))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuHt_wmean':
        ax.hexbin(stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['BuHt_wmean'], stn_buffers[(stn_buffers['BuHt_wmean']!=0)&(stn_buffers['BuHt_wmean']<20)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars[vars['BuHt_wmean']>1]['BuHt_wmean'], vars[vars['BuHt_wmean']>1]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height ($\it{H}$, m)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.72""\n""$ρ_{mob,300,T}$: 0.58", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuHt_wmean'], is_sorted=False, frac =0.7)
        print(params[i], calculate_slopes(temp_lowess, turn_a=6, turn_b=7, x_max=15))
        ax.plot(temp_lowess[:-20, 0], temp_lowess[:-20, 1], color='blue', linewidth=2)
        
    elif params[i] == 'BuW_delaunay_mean_merged':
        ax.hexbin((stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['BuW_delaunay_mean_merged']), stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter((vars['BuW_delaunay_mean_merged']), vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Width ($\it{W}$, m)',fontsize=16)
        #ax.set_xscale('log')
        ax.set_xlim(0,250)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.15""\n""$ρ_{mob,300,T}$: -0.28", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))      
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['BuW_delaunay_mean_merged'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=150, turn_b=170))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'SVF_3D_mean':
        ax.hexbin(stn_buffers['SVF_3D_mean'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['SVF_3D_mean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Sky View Factor ($\it{SVF}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.64""\n""$ρ_{mob,300,T}$: -0.71", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['SVF_3D_mean'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.8, turn_b=0.9))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    elif params[i] == 'HW_del_merged':
        ax.hexbin(stn_buffers[stn_buffers['HW_del_merged']!=0]['HW_del_merged'], stn_buffers[stn_buffers['HW_del_merged']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_del_merged'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v1 ($\it{H:W_1}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.45""\n""$ρ_{mob,300,T}$: 0.56", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))   
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_del_merged'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.5, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'HW_geo':
        ax.hexbin(stn_buffers[stn_buffers['HW_geo']!=0]['HW_geo'], stn_buffers[stn_buffers['HW_geo']!=0]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['HW_geo'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v2 ($\it{H:W_2}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.69""\n""$ρ_{mob,300,T}$: 0.72", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['HW_geo'], is_sorted=False)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.4, turn_b=0.8))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
    
    elif params[i] == 'StrHW_wmean':
        ax.hexbin(stn_buffers[stn_buffers['StrHW_wmean']<10]['StrHW_wmean'],stn_buffers[stn_buffers['StrHW_wmean']<10]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrHW_wmean'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Height:Width v3 ($\it{H:W_3}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.61""\n""$ρ_{mob,300,T}$: 0.51", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrHW_wmean'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.6, turn_b=0.7))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        #ax.set_xlim(0,1.5)

    elif params[i] == 'BuERI_mode_merged':
        # swap order so that low ERI is high SC 
        ax.hexbin(1-stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['BuERI_mode_merged'], stn_buffers[(stn_buffers['BuERI_mode_merged']>0)&(stn_buffers['BuERI_mode_merged']<1)]['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['BuERI_mode_merged'], vars[(vars['BuERI_mode_merged']>0)&(vars['BuERI_mode_merged']<1)]['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Shape Complexity ($\it{SC}$)',fontsize=16)
        ax.text(0.05, 0.05, r"$ρ_{fix,300,T}$: -0.68""\n""$ρ_{mob,300,T}$: -0.66", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuERI_mode_merged'], is_sorted=False, frac =0.9)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.45, turn_b=0.55))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    elif params[i] == 'BuAdj':
        ax.hexbin(1-stn_buffers['BuAdj'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(1-vars['BuAdj'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Adjacency ($\it{Adj}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: -0.63""\n""$ρ_{mob,300,T}$: 0.61", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=1-stn_buffers['BuAdj'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.42, turn_b=0.52))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)
        
    elif params[i] == 'StrClo400_median':
        ax.hexbin(stn_buffers['StrClo400_median'], stn_buffers['temp_no_cooling']+0.2,mincnt=1, gridsize=50, cmap='gist_heat_r', vmin=-2,vmax=45)
        ax.scatter(vars['StrClo400_median'], vars['2023-06-27 20:00:00+00:00'], marker ='x', c='darkslategrey', alpha =1,s=100,linewidths=2)
        ax.set_xlabel('Street Network Connectivity ($\it{C}$)',fontsize=16)
        ax.text(0.58, 0.05, r"$ρ_{fix,300,T}$: 0.29""\n""$ρ_{mob,300,T}$: 0.40", transform=ax.transAxes, fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", edgecolor='grey', facecolor='none'))
        temp_lowess = lowess(endog=stn_buffers['temp_no_cooling']+0.2, exog=stn_buffers['StrClo400_median'], is_sorted=False, frac =0.95)
        print(params[i], calculate_slopes(temp_lowess, turn_a=0.000008, turn_b=0.000013))
        ax.plot(temp_lowess[:, 0], temp_lowess[:, 1], color='blue', linewidth=2)

    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel(r'$\it{T_a}$ ($\degree$C)',fontsize=16)
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks
    
# Add a colorbar below the entire plot
cbar_ax = fig.add_axes([0.2, 0.03, 0.6, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', drawedges=False)

# Customize the colorbar
cbar.ax.tick_params(axis='x', labelsize=14)  # Adjust tick font size
cbar.ax.set_xlabel('Count', fontsize=16)  # Set the label

plt.tight_layout(w_pad=0.01, h_pad=0.3)

plt.savefig('../../figures/fig6/fig6_september.png', dpi=650, bbox_inches='tight')
plt.savefig('../../figures/fig6/fig6_september.pdf')
plt.savefig('../../figures/fig6/fig6_september.svg')
plt.show()

In [ ]:
len(temp_lowess)

In [ ]:
data = stn_buffers[[i,'temp_no_cooling']].dropna()
x = data[i].values
y = data['temp_no_cooling'].values + 0.2
pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
print(pw_fit.summary())                                 

In [ ]:
params = ['BuAre_sum']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pw_fit.plot_fit(color="red", linewidth=4)

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
# with bootstrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import piecewise_regression

# Configuration
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
n_iterations = 5  # Start with 100, increase to 500-1000 for final paper
system_ids = stn_buffers['System_ID'].unique()

for i in params:
    print(f"Analyzing: {i}")
    
    # --- Bootstrap Loop ---
    boot_breakpoints = []
    
    for b in range(n_iterations):
        # 1. Resample System_IDs with replacement
        resampled_ids = np.random.choice(system_ids, size=len(system_ids), replace=True)
        
        # 2. Filter dataframe to only include these routes
        # This preserves the correlation within each route
        boot_df = stn_buffers[stn_buffers['System_ID'].isin(resampled_ids)].copy()
        
        # Apply your specific parameter filters
        if i == 'BuW_delaunay_mean_merged':
            boot_df = boot_df[boot_df['BuW_delaunay_mean_merged'] < 250]
        
        # Prepare x and y
        data_clean = boot_df[[i, 'temp_no_cooling']].dropna()
        
        if len(data_clean) < 10: # Safety check for empty resamples
            continue
            
        if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
            x_boot = (1 - data_clean[i]).values
        else:
            x_boot = data_clean[i].values
            
        y_boot = data_clean['temp_no_cooling'].values + 0.2

        try:
            # Fit model and extract the breakpoint
            pw_fit_boot = piecewise_regression.Fit(x_boot, y_boot, n_breakpoints=1)
            # Assuming the breakpoint is stored in the model's breakpoints_ attribute
            boot_breakpoints.append(pw_fit_boot.breakpoints[0])
        except:
            # Catch cases where the model fails to converge on a specific resample
            continue

    # Calculate Conservative Confidence Interval (2.5% and 97.5% percentiles)
    ci_low = np.percentile(boot_breakpoints, 2.5)
    ci_high = np.percentile(boot_breakpoints, 97.5)
    median_bp = np.median(boot_breakpoints)
    
    print(f"Bootstrap Breakpoint: {median_bp:.4f} (95% CI: {ci_low:.4f} - {ci_high:.4f})")

    # --- Final Plotting (using the original full dataset) ---
    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged'] < 250][[i, 'temp_no_cooling']].dropna()
    else: 
        data = stn_buffers[[i, 'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1 - data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    
    plt.figure(figsize=(8, 6))
    pw_fit.plot_data(color="grey", s=20)
    pw_fit.plot_fit(color="red", linewidth=4)
    
    # Instead of the default CI, plot the bootstrap median and CI
    plt.axvline(median_bp, color='blue', linestyle='-', label=f'Boot BP: {median_bp:.3f}')
    plt.axvline(ci_low, color='blue', linestyle='--', label=f'95% CI: {ci_low:.3f} - {ci_high:.3f}')
    plt.axvline(ci_high, color='blue', linestyle='--')
    
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.legend()
    plt.show()

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<100][[i,'temp_no_cooling']].dropna()

    if i == 'BuHt_wmean':
        data = stn_buffers[stn_buffers['BuHt_wmean']<20][[i,'temp_no_cooling']].dropna()

    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    

    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
fig, axs = plt.subplots(2,1, figsize=(10,13))

linestyle_mapping = {
    'BuAre_sum': '-',
    'BuVol_3D_sum': '-',
    'BuEWA_3D_sum': '-',
    'HW_geo':'-',
    'BuAdj':'-',
    'HW_del_merged':'-',
    'BuHt_wmean': '-',
    'StrHW_wmean': '--',
    'SVF_3D_mean': '--',
    'BuERI_mode_merged': '-',
    'StrClo400_median': '--',
    'BuW_delaunay_mean_merged':'-'
}

color_mapping = {
    'BuAre_sum': 'skyblue',
    'BuVol_3D_sum': 'royalblue',
    'BuEWA_3D_sum': 'darkblue',
    'BuSWR_3D_median':'tab:purple',
    'BuHt_wmean': '#9467BD',
    'StrHW_wmean': '#F7B6D2',
    'HW_geo': '#E377C2',
    'HW_del_merged': '#D62728',
    'BuW_delaunay_mean_merged': 'gold',
    'SVF_3D_mean': 'tab:green',
    'BuERI_mode_merged': 'black',
    'BuAdj':'silver',
    'StrClo400_median': 'grey'
}

var_name_mapping = {
    'BuAre_sum': 'Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',
    'BuVol_3D_sum': 'Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',
    'BuEWA_3D_sum': 'Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)', 
    'BuAdj': 'Adjacency ($\it{Adj}$)',
    'BuHt_wmean': 'Height ($\it{H}$, m)',
    'SVF_3D_mean': 'Sky View Factor ($\it{SVF}$)',
    'BuERI_mode_merged': 'Shape Complexity ($\it{SC}$)',
    'StrClo400_median': 'Street Network Connecitvity ($\it{C}$)',
    'StrHW_wmean': 'Height:Width v3 ($\it{H:W_3}$)',
    'HW_geo':'Height:Width v2 ($\it{H:W_2}$)',
    'HW_del_merged':'Height:Width v1 ($\it{H:W_1}$)',
    'BuW_delaunay_mean_merged':'Width ($\it{W}$, m)',
    'HW_knn1_merged':'HW_knn1_merged',
    'HW_knn5_merged':'HW_knn5_merged',
    'NdDen':'$\it{ND}$'
    }

axs[0].set_xscale('log')
for param in params:
    axs[0].plot(combined_stats[param+'_spearman'], label=var_name_mapping[param], linestyle=linestyle_mapping[param], linewidth=2, color=color_mapping[param])
axs[0].set_ylabel(r"$\rho_{\mathrm{fix},r,ΔT}$", fontsize=20)
axs[0].set_ylim(0, 1)
axs[0].set_xlim(30, 2000)
#axs[0, 0].set_xticklabels([])
axs[0].tick_params(axis='both', which='major', labelsize=14)
axs[0].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
axs[0].get_xaxis().set_minor_formatter(matplotlib.ticker.ScalarFormatter())
axs[0].grid()
axs[0].tick_params(axis='x', which='minor', rotation=45,labelsize=12)
axs[0].tick_params(axis='x', labelrotation=45)


axs[1].set_xscale('log')
for param in params:
    axs[1].plot(abs(df[param+'_spearman_corr']), label=var_name_mapping[param], linestyle=linestyle_mapping[param], linewidth=2, color=color_mapping[param])
axs[1].set_ylabel(r"$\rho_{\mathrm{mob},r,T}$", fontsize=20)
axs[1].set_ylim(0, 1)
axs[1].set_xlim(30, 2000)

axs[1].get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
axs[1].get_xaxis().set_minor_formatter(matplotlib.ticker.ScalarFormatter())
axs[1].tick_params(axis='both', which='major', labelsize=14)
axs[1].grid()
axs[1].set_xlabel('Radius (m)', fontsize=20)
axs[1].tick_params(axis='x', labelrotation=45)
axs[1].tick_params(axis='x', which='minor', rotation=45,labelsize=12)
#axs[1].legend(['$\it{A_B}$','$\it{V_B}$','$\it{A_F}$','$\it{IBD}$', '$\it{Adj}$', '$\it{SWR}$', '$\it{H_B^A}$', '$\it{H/W}$', '$\it{SVF}$', '$\it{ERI}$', '$\it{C}$'],prop={'size': 11}, bbox_to_anchor=(0.01, 1), loc='upper left', title='  Featured\nParameters', title_fontsize=12)
axs[1].legend(prop={'size': 11}, bbox_to_anchor=(0.5, -0.4), loc='center', title_fontsize=12, ncol=2)

plt.tight_layout(pad = 2, w_pad=1)
plt.savefig('../../figures/fig7/spear_new_new.png',dpi=300, bbox_inches='tight')
plt.savefig('../../figures/fig7/spear_new_new.pdf')
plt.savefig('../../figures/fig7/spear_new_new.svg')
plt.show()

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if i = 'BuERI_mode_merged' or i == 'BuAdj':
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
for i in range(12):  # First 12 subplots

    if params[i] == 'BuAre_sum':
        data= stn_buffers[['BuAre_sum','temp_no_cooling']].dropna()

        x = data['BuAre_sum'].values
        y = data['temp_no_cooling'].values + 0.2


        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        pw_fit.summary()

        # Plot the data, fit, breakpoints and confidence intervals
        pw_fit.plot_data(color="grey", s=20)
        # Pass in standard matplotlib keywords to control any of the plots
        pw_fit.plot_fit(color="red", linewidth=4)
        pw_fit.plot_breakpoints()
        pw_fit.plot_breakpoint_confidence_intervals()
        plt.xlabel("x")
        plt.ylabel("y")
        plt.show()

In [ ]:
np.isfinite(x)

In [ ]:
plt.scatter(x,y)

In [ ]:
x_normalized

In [ ]:
len(stn_buffers['BuAre_sum'].values)

In [ ]:
len(stn_buffers['temp_no_cooling'].values+0.2)

In [ ]:
stn_buffers['temp_no_cooling'].values

### Calculate labels for plots

In [ ]:
# spearman correlation of data
from scipy.stats import pearsonr, spearmanr
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

for i in params:

    data = stn_buffers[[i,'Temperature_diff_K']].dropna()
    spearman = spearmanr(data[i], data['Temperature_diff_K'])
    print(i, spearman)


## For all spatial scales

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

for i in params:

    data = stn_buffers[[i,'Temperature_diff_K']].dropna()
    spearman = spearmanr(data[i], data['Temperature_diff_K'])
    print(i, spearman)

In [ ]:
# MI
from sklearn.feature_selection import mutual_info_regression

for i in params:

    data = stn_buffers[[i,'Temperature_diff_K']].dropna()
    mi = mutual_info_regression(data[[i]], data['Temperature_diff_K'])
    print(i, mi)


In [ ]:
# linear regression with statsmodels
import statsmodels.api as sm
for i in params:

    data = stn_buffers[[i,'Temperature_diff_K']].dropna()
    X = sm.add_constant(data[i])  # Add a constant term for the intercept
    model = sm.OLS(data['Temperature_diff_K'], X).fit()
    print(i, model.summary())


### For all parameters (and table 2)

In [ ]:
spearman_df = {}

param_list = []
spearman_r_list = []
spearman_p_list = []

for i in stn_buffers.drop(['station_id','System_ID','Time_UTC','Seconds_in_day','Altitude','Latitude','Longitude','Temperature_diff_K','Absolute_temperature_degC','Relhumidity_diff_percent','Relhumidity_percent','Vapourpressure_diff_hPa','Vapourpressure_hPa','geometry','SVF_3D'], axis=1).columns:
    print(i)
    data = stn_buffers[[i,'Temperature_diff_K']].dropna()
    spearman_r, spearman_p = spearmanr(data[i], data['Temperature_diff_K'])
    param_list.append(i)
    spearman_r_list.append(spearman_r)
    spearman_p_list.append(spearman_p)

spearman_df['parameter'] = param_list
spearman_df['spearman_r'] = spearman_r_list
spearman_df['spearman_p'] = spearman_p_list

spearman_df = pd.DataFrame(spearman_df)


In [117]:
# Define mapping of abbreviations to categories
# Define mapping of abbreviations to categories
prefix_to_category = {
    'BuAre': 'Dimension', 'BuHt': 'Dimension', 'BuPer': 'Dimension',
    'BuLAL': 'Dimension', 'BuCCD': 'Dimension', 'BuCor': 'Dimension',
    'BuSA' : 'Dimension', 'BuVol': 'Dimension', 'BuSWA': 'Dimension', 
    'BuEWA': 'Dimension','BumVE' : 'Dimension',  'BuMVE' : 'Dimension',
    'CyAre': 'Dimension', 'CyInd': 'Dimension', 'BuCAR': 'Dimension', 
    'BuCWA': 'Shape', 'BuCorDev': 'Shape',
    'BuCon': 'Shape', 'BuElo': 'Shape', 'BuERI': 'Shape', 'BuEPI': 'Shape',
    'BuFR': 'Shape', 'BuFF': 'Shape', 'BuFD': 'Shape', 'BuCf': 'Shape',
    'BuRec': 'Shape', 'BuShI': 'Shape', 'BuSqC': 'Shape', 'BuEx': 'Shape',
    'BuSqu': 'Shape', 'BuCir': 'Shape','BuCCo': 'Shape', 'BuHem': 'Shape',
    'BuFra' : 'Shape','BuCube' : 'Shape', 'BuCubo' : 'Shape',  'BuProx' : 'Shape',
    'BuSpi' : 'Shape', 'BuPerC' : 'Shape', 'BuGir' : 'Shape', 'BuDisp' : 'Shape',
    'BuRan' : 'Shape', 'BuDep' : 'Shape', 'BuAdj': 'Distribution', 'BuIBD': 'Distribution',
    'BuSWR': 'Distribution', 'BuOri': 'Orientation', 'BuAli': 'Orientation',
    'StrAli': 'Orientation', 'StrW': 'Distribution', 'StrWD': 'Distribution',
    'StrOpe': 'Distribution', 'StrHW': 'Distribution', 'StrH': 'Dimension',
    'StrHD': 'Distribution','StrLen': 'Dimension',
    'StrCNS': 'Dimension', 'BpM': 'Intensity', 'StrLin': 'Shape',
    'StrClo400': 'Connectivity', 'StrBet400': 'Connectivity', 
    'StrSCl': 'Connectivity', 'StrCyc400': 'Connectivity', 
    'StrENR400': 'Connectivity', 'StrGam400': 'Connectivity', 
    'StrDeg': 'Connectivity', 'StrMes400': 'Connectivity',
    'SVF': 'Distribution'
}

unique_prefixes = [item.split('_')[0] for item in spearman_df['parameter'].values]

# Generate categories data dynamically
categories_data = [
    {'Category': prefix_to_category.get(prefix, 'Unknown'), 'Abbrev.': spearman_df['parameter'].values[i]}
    for i,prefix in enumerate(unique_prefixes)
]

In [118]:
# add category column to spearman_df
spearman_df['Category'] = [prefix_to_category.get(prefix, 'Unknown') for prefix in unique_prefixes]

In [ ]:
for i in spearman_df.sort_values(by='spearman_r', ascending=False).values:
    print(i)

In [120]:
spearman_df.to_csv('../../data/processed_data/mbike/270623_spearman_correlation.csv')
spearman_df.to_excel('../../data/processed_data/mbike/270623_spearman_correlation.xlsx')

### MI (Mutual Information)

In [ ]:
mi_df = {}

mi_param_list = []
mi_val_list = []

for i in stn_buffers.drop(['station_id','System_ID','Time_UTC','Seconds_in_day','Altitude','Latitude','Longitude','Temperature_diff_K','Absolute_temperature_degC','Relhumidity_diff_percent','Relhumidity_percent','Vapourpressure_diff_hPa','Vapourpressure_hPa','geometry','SVF_3D'], axis=1).columns:
    print(i)
    X = stn_buffers[[i]]
    y = stn_buffers['Temperature_diff_K']

    # Drop rows with NaN values in X and y
    X = X.dropna()
    y = y.loc[X.index]

    if len(X) > 2:
        mi = mutual_info_regression(X, y)
    else:
        mi = [np.nan]
    mi_param_list.append(i)
    mi_val_list.append(mi[0])

mi_df['parameter'] = mi_param_list
mi_df['mi'] = mi_val_list

mi_df = pd.DataFrame(mi_df)


In [82]:
# Define mapping of abbreviations to categories
# Define mapping of abbreviations to categories
prefix_to_category = {
    'BuAre': 'Dimension', 'BuHt': 'Dimension', 'BuPer': 'Dimension', 'BuNum':'Dimension',
    'BuLAL': 'Dimension', 'BuCCD': 'Dimension', 'BuCor': 'Dimension',
    'BuSA' : 'Dimension', 'BuVol': 'Dimension', 'BuSWA': 'Dimension', 
    'BuEWA': 'Dimension','BumVE' : 'Dimension',  'BuMVE' : 'Dimension',
    'CyAre': 'Dimension', 'CyInd': 'Dimension', 'BuCAR': 'Dimension', 'BuSurf': 'Dimension',
    'BuCWA': 'Shape', 'BuCorDev': 'Shape', 'BuRough': 'Shape',
    'BuCon': 'Shape', 'BuElo': 'Shape', 'BuERI': 'Shape', 'BuEPI': 'Shape',
    'BuFR': 'Shape', 'BuFF': 'Shape', 'BuFD': 'Shape', 'BuCf': 'Shape',
    'BuRec': 'Shape', 'BuShI': 'Shape', 'BuSqC': 'Shape', 'BuEx': 'Shape',
    'BuSqu': 'Shape', 'BuCir': 'Shape','BuCCo': 'Shape', 'BuHem': 'Shape',
    'BuFra' : 'Shape','BuCube' : 'Shape', 'BuCubo' : 'Shape',  'BuProx' : 'Shape',
    'BuSpi' : 'Shape', 'BuPerC' : 'Shape', 'BuGir' : 'Shape', 'BuDisp' : 'Shape',
    'BuRan' : 'Shape', 'BuDep' : 'Shape', 'BuAdj': 'Distribution', 'BuIBD': 'Distribution',
    'BuSWR': 'Distribution', 'BuOri': 'Orientation', 'BuAli': 'Orientation',
    'StrAli': 'Orientation', 'StrW': 'Distribution', 'StrWD': 'Distribution',
    'StrOpe': 'Distribution', 'StrHW': 'Distribution', 'StrH': 'Dimension',
    'StrHD': 'Distribution','StrLen': 'Dimension',
    'StrCNS': 'Dimension', 'BpM': 'Intensity', 'StrLin': 'Shape',
    'StrClo400': 'Connectivity', 'StrBet400': 'Connectivity', 
    'StrSCl': 'Connectivity', 'StrCyc400': 'Connectivity', 
    'StrENR400': 'Connectivity', 'StrGam400': 'Connectivity', 
    'StrDeg': 'Connectivity', 'StrMes400': 'Connectivity',
    'SVF': 'Distribution'
}

unique_prefixes = [item.split('_')[0] for item in mi_df['parameter'].values]

# Generate categories data dynamically
categories_data = [
    {'Category': prefix_to_category.get(prefix, 'Unknown'), 'Abbrev.': mi_df['parameter'].values[i]}
    for i,prefix in enumerate(unique_prefixes)
]

In [83]:
# add category column to spearman_df
mi_df['Category'] = [prefix_to_category.get(prefix, 'Unknown') for prefix in unique_prefixes]

In [ ]:
for i in mi_df.sort_values(by='mi', ascending=False).values:
    print(i)

In [85]:
mi_df.to_csv('../../data/processed_data/mbike/270623_mutual_info.csv')
mi_df.to_excel('../../data/processed_data/mbike/270623_mutual_info.xlsx')